In [25]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import joblib

In [4]:
df=pd.read_csv('telco_customer_cleaned')

In [5]:
df.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False


In [7]:
X=df.drop('Churn',axis=1)
y=df['Churn']

In [9]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [10]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.fit_transform(X_test)

In [29]:
models={
    "Logistic Regression":LogisticRegression(class_weight='balanced',max_iter=1000),
    "Random Forest":RandomForestClassifier(n_estimators=200,random_state=42),
    "XGBoost":XGBClassifier(use_label_encoder=False,eval_metric='logloss')
}

results=[]
for name,model in models.items():
    if name=="Logistic Regression":
        model.fit(X_train_scaled,y_train)
        y_prob=model.predict_proba(X_test_scaled)[:,1]
        threshold=0.3
        y_pred=(y_prob>=threshold).astype(int)
    else:
        model.fit(X_train,y_train)
        y_prob=model.predict_proba(X_test)[:,1]
        threshold=0.3
        y_pred=(y_prob>=threshold).astype(int)

    file_name = "".join(word.capitalize() for word in name.split()) + ".pkl"
    joblib.dump(model,file_name)

    acc=accuracy_score(y_test,y_pred)
    recall=recall_score(y_test,y_pred)
    print(f"\n{name}")
    print(f"Accuracy:",acc)
    print(classification_report(y_test,y_pred))
    results.append((name,acc,recall))
result_df=pd.DataFrame(results,columns=['Model','Accuracy','Recall_class_1'])
print("\nModel comparision:\n",result_df)


Logistic Regression
Accuracy: 0.6515259048970902
              precision    recall  f1-score   support

           0       0.96      0.55      0.70      1036
           1       0.43      0.94      0.59       373

    accuracy                           0.65      1409
   macro avg       0.70      0.75      0.64      1409
weighted avg       0.82      0.65      0.67      1409


Random Forest
Accuracy: 0.7629524485450674
              precision    recall  f1-score   support

           0       0.89      0.77      0.83      1036
           1       0.54      0.75      0.63       373

    accuracy                           0.76      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.76      0.77      1409



C:\Users\kavyam mikul shah\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:200: UserWarning: [09:09:49] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost
Accuracy: 0.772888573456352
              precision    recall  f1-score   support

           0       0.89      0.79      0.84      1036
           1       0.55      0.73      0.63       373

    accuracy                           0.77      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.77      0.78      1409


Model comparision:
                  Model  Accuracy  Recall_class_1
0  Logistic Regression  0.651526        0.943700
1        Random Forest  0.762952        0.747989
2              XGBoost  0.772889        0.726542


In [28]:
joblib.dump(scaler,"scaler.pkl")
joblib.dump(X.columns.to_list(),"features.pkl")

['features.pkl']